In [ ]:
# Install required libraries
!pip install -q nba_api pandas-gbq google-cloud-bigquery

In [ ]:
import os
import time
import pandas as pd
from google.cloud import bigquery
from nba_api.stats.endpoints import playerindex, leaguedashplayerbiostats, leaguegamefinder
from nba_api.stats.static import teams

# Phase 2: Fact & Dimension Table Generation (Data Warehouse Pipeline)
This notebook orchestrates the extraction of NBA player and team metadata from the NBA API, and transforms raw play-by-play data into a structured dimensional data warehouse in BigQuery.

### Analytical Methodology
- **Fact Tables:** `fct_shots` and `fct_fouls` are generated by deduplicating and filtering raw event logs. We join shooting events with foul events using a 2-second proximity window to accurately map drawn fouls to physical shot attempts.
- **Dimension Tables:** Player biographies, game schedules, and team metadata are extracted to support analytical grouping and geographic color-mapping.

### Prerequisites
- A Google Cloud Project with BigQuery enabled and configured via environment variables.

In [ ]:
# --- GLOBAL CONFIGURATION ---
# Use environment variables for BigQuery Project IDs to ensure reproducibility across different environments
PROJECT_ID = os.environ.get("BQ_PROJECT_ID", "mapping-nba-fouls")
DATASET_ID = os.environ.get("BQ_DATASET_ID", "capstone_project")
LOCATION = "US"

# Define seasons for API extraction
SEASONS_API = ["2019-20", "2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]

print(f"Target Project: {PROJECT_ID}")
print(f"Target Dataset: {DATASET_ID}")

In [ ]:
# Instantiate BigQuery client for data warehouse operations
client = bigquery.Client(project=PROJECT_ID)

# --- TABLE 1: FACT SHOTS (fct_shots) ---
# Purpose: Isolates physical shot attempts and joins them with shooting foul events.
# Logic: A 2-second game-clock window is used to map fouls to their corresponding physical shot attempts.
sql_fct_shots_final = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.fct_shots`
CLUSTER BY season_id, player_id, is_foul_drawn
AS
WITH physical_shots AS (
  SELECT
    event_num, season_id, game_id, period, game_clock_seconds, game_clock,
    player_id, team_id, loc_x, loc_y, (event_type = '1') as is_made
  FROM `{PROJECT_ID}.{DATASET_ID}.raw_shufinskiy_datanba`
  WHERE event_type IN ('1', '2') AND loc_y < 430
),
shooting_fouls AS (
  SELECT
    event_num, season_id, game_id, period, game_clock_seconds,
    player_id, loc_x, loc_y
  FROM `{PROJECT_ID}.{DATASET_ID}.raw_shufinskiy_datanba`
  WHERE event_type = '6' AND msg_type = '2' AND loc_y < 430
),
joined_physical AS (
  SELECT
    s.*,
    IF(f.player_id IS NOT NULL, TRUE, FALSE) as is_foul_drawn,
    f.event_num as foul_event_num,
    'Physical Shot' as attempt_type,
    -- RANK THE MATCHES: If one shot matches 2 fouls, row_number 1 is the closest in time
    ROW_NUMBER() OVER(
        PARTITION BY s.game_id, s.event_num
        ORDER BY ABS(s.game_clock_seconds - f.game_clock_seconds) ASC
    ) as match_rank
  FROM physical_shots s
  LEFT JOIN shooting_fouls f
    ON s.game_id = f.game_id
    AND s.player_id = f.player_id
    AND ABS(s.game_clock_seconds - f.game_clock_seconds) <= 2
)

-- 1. Grab only the BEST match for physical shots
SELECT * EXCEPT(match_rank) FROM joined_physical WHERE match_rank = 1

UNION ALL

-- 2. Fouls on Misses (Deduplicated by logic: only those with NO match)
SELECT
    f.event_num, f.season_id, f.game_id, f.period, f.game_clock_seconds,
    NULL as game_clock, f.player_id, NULL as team_id, f.loc_x, f.loc_y,
    FALSE as is_made, TRUE as is_foul_drawn, f.event_num as foul_event_num,
    'Foul on Miss' as attempt_type
FROM shooting_fouls f
LEFT JOIN physical_shots s
  ON f.game_id = s.game_id
  AND f.player_id = s.player_id
  AND ABS(f.game_clock_seconds - s.game_clock_seconds) <= 2
WHERE s.player_id IS NULL;
"""

print("Finalizing deduplicated fct_shots...")
client.query(sql_fct_shots_final).result()
print("✅ Success: Table fct_shots is now 100% unique.")

In [ ]:
# --- TABLE 2: FACT FOULS (fct_fouls) ---
# Purpose: Creates a targeted table for geospatial foul mapping.
# Logic: Filters the raw event log exclusively for Personal (1) and Shooting (2) fouls.
sql_fct_fouls = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.fct_fouls`
CLUSTER BY season_id, player_id, foul_category
AS
SELECT
    event_num,
    season_id,
    game_id,
    period,
    game_clock,
    game_clock_seconds,
    player_id,
    team_id,
    opp_player_id as fouled_player_id,
    loc_x,
    loc_y,
    (msg_type = '2') as is_shooting_foul,
    CASE
        WHEN msg_type = '2' THEN 'Shooting'
        WHEN msg_type = '1' THEN 'Personal'
        ELSE 'Other' -- Protective fallback, though the WHERE clause filters this
    END AS foul_category,
    description
FROM `{PROJECT_ID}.{DATASET_ID}.raw_shufinskiy_datanba`
WHERE event_type = '6'
  AND msg_type IN ('1', '2')
  AND loc_y < 430;
"""

print("Generating fct_fouls (Personal & Shooting only)...")
client.query(sql_fct_fouls).result()
print("✅ Table fct_fouls updated and focused.")

In [ ]:
# --- AUDIT: FACT SHOTS INTEGRITY ---
# Verify the aggregation logic and calculate the baseline metrics for true shot attempts.
verify_query = f"""
SELECT
    attempt_type,
    is_foul_drawn,
    COUNT(*) as total_count,
    COUNT(DISTINCT player_id) as unique_players,
    ROUND(AVG(loc_y), 2) as avg_dist_y
FROM `{PROJECT_ID}.{DATASET_ID}.fct_shots`
GROUP BY 1, 2
ORDER BY 1, 2
"""

print("Auditing fct_shots integrity...")
audit_df = client.query(verify_query).to_dataframe()
display(audit_df)

# Check for the total count of 'True Shot Attempts'
print(f"\nTotal True Shot Attempts: {audit_df['total_count'].sum():,}")

In [ ]:
# --- AUDIT: DUPLICATE RESOLUTION ---
# Validates that the proximity join logic did not generate duplicate shot events within a single game.
correct_dup_query = f"""
SELECT game_id, event_num, COUNT(*) as occurrence
FROM `{PROJECT_ID}.{DATASET_ID}.fct_shots`
GROUP BY 1, 2
HAVING occurrence > 1
"""

print("Checking for true duplicates (Game ID + Event Num)...")
true_dups = client.query(correct_dup_query).to_dataframe()

if true_dups.empty:
    print("✅ Success: Every shot attempt is unique within its game. No data double-counting.")
else:
    print(f"⚠️ Warning: Found {len(true_dups)} instances where a shot record was duplicated within a game.")
    display(true_dups.head(15))

In [ ]:
# --- DATA INVENTORY AUDIT ---
inventory_query = f"""
WITH stats AS (
  -- Raw Ingested Table
  SELECT
    'raw_shufinskiy_datanba' as table_name,
    COUNT(*) as record_count,
    (SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS` WHERE table_name = 'raw_shufinskiy_datanba') as attribute_count
  FROM `{PROJECT_ID}.{DATASET_ID}.raw_shufinskiy_datanba`

  UNION ALL

  -- True Shot Attempts Table
  SELECT
    'fct_shots' as table_name,
    COUNT(*) as record_count,
    (SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS` WHERE table_name = 'fct_shots') as attribute_count
  FROM `{PROJECT_ID}.{DATASET_ID}.fct_shots`

  UNION ALL

  -- Focused Fouls Table
  SELECT
    'fct_fouls' as table_name,
    COUNT(*) as record_count,
    (SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS` WHERE table_name = 'fct_fouls') as attribute_count
  FROM `{PROJECT_ID}.{DATASET_ID}.fct_fouls`
)
SELECT * FROM stats
"""

inventory_df = client.query(inventory_query).to_dataframe()

print("--- Final Data Inventory (Phase 1 & 2) ---")
display(inventory_df)

# Calculate the "Signal Retention"
total_raw = inventory_df.loc[inventory_df['table_name'] == 'raw_shufinskiy_datanba', 'record_count'].values[0]
total_shots = inventory_df.loc[inventory_df['table_name'] == 'fct_shots', 'record_count'].values[0]

print(f"\nAnalysis Note: You are focusing on {total_shots:,} shot attempts,")
print(f"which represents { (total_shots/total_raw)*100:.1f}% of your total play-by-play data.")

In [ ]:
# --- DIMENSION: PLAYER SEASONS (dim_player_seasons) ---
# Purpose: Extracts biographical and positional metadata from the NBA API.
# Note: Relies on the leaguedashplayerbiostats and playerindex endpoints.

all_season_bios = []

print("Starting season-specific bio data collection...")

for s_string in SEASONS_API:
    print(f"Fetching bio stats for {s_string}...")
    try:
        bio = leaguedashplayerbiostats.LeagueDashPlayerBioStats(season=s_string).get_data_frames()[0]
        bio['api_season_id'] = s_string
        year_part = s_string.split('-')[0]
        bio['season_id_reg'] = '2' + year_part
        bio['season_id_post'] = '4' + year_part
        bio = bio.rename(columns={'PLAYER_ID': 'player_id'})
        all_season_bios.append(bio)
        time.sleep(1)
    except Exception as e:
        print(f"Error fetching {s_string}: {e}")

df_bios_long = pd.concat(all_season_bios, ignore_index=True)

print("Fetching player positions and static metadata...")
p_index_df = playerindex.PlayerIndex(season='2024-25').get_data_frames()[0]
p_index_df = p_index_df.rename(columns={'PERSON_ID': 'player_id', 'POSITION': 'position'})

reg_df = df_bios_long.copy().drop(columns=['season_id_post']).rename(columns={'season_id_reg': 'season_id'})
post_df = df_bios_long.copy().drop(columns=['season_id_reg']).rename(columns={'season_id_post': 'season_id'})
dim_player_seasons = pd.concat([reg_df, post_df], ignore_index=True)

dim_player_seasons['player_id'] = dim_player_seasons['player_id'].astype(str)
p_index_df['player_id'] = p_index_df['player_id'].astype(str)

dim_player_seasons = dim_player_seasons.merge(
    p_index_df[['player_id', 'position']],
    on='player_id',
    how='left'
)

dim_player_seasons = dim_player_seasons.rename(columns={
    'PLAYER_NAME': 'full_name',
    'TEAM_ABBREVIATION': 'team_at_time',
    'PLAYER_HEIGHT_INCHES': 'height_inches',
    'PLAYER_WEIGHT': 'weight_lbs',
    'COLLEGE': 'school',
    'COUNTRY': 'country',
    'DRAFT_YEAR': 'draft_year',
    'DRAFT_ROUND': 'draft_round',
    'DRAFT_NUMBER': 'draft_number'
})

final_cols = [
    'player_id', 'season_id', 'full_name', 'team_at_time', 'position',
    'height_inches', 'weight_lbs', 'country', 'school',
    'draft_year', 'draft_round', 'draft_number'
]

dim_player_seasons = dim_player_seasons[final_cols]

TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.dim_player_seasons"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    clustering_fields=["player_id", "season_id"]
)

print(f"Loading {len(dim_player_seasons)} records into BigQuery...")
client.load_table_from_dataframe(dim_player_seasons, TABLE_ID, job_config=job_config).result()

print("✅ dim_player_seasons loaded.")

In [ ]:
# --- DIMENSION: GAMES (dim_games) ---
# Fetch schedule and game outcomes to support temporal filtering (e.g., Regular Season vs. Playoffs).
all_games = []
for s in SEASONS_API:
    finder = leaguegamefinder.LeagueGameFinder(season_nullable=s, league_id_nullable='00')
    all_games.append(finder.get_data_frames()[0])

df_games = pd.concat(all_games, ignore_index=True)

# Standardize schema and flag playoff games based on the NBA ID prefix ('004')
dim_games = df_games[['GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL']].copy()
dim_games.columns = ['game_id', 'game_date', 'matchup', 'result']
dim_games['is_playoff'] = dim_games['game_id'].str.startswith('004')

# Load structured game data into BigQuery
client.load_table_from_dataframe(
    dim_games,
    f"{PROJECT_ID}.{DATASET_ID}.dim_games",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
).result()

print("✅ dim_games loaded.")

In [ ]:
# --- DIMENSION: TEAMS (dim_teams) ---
# Fetch static team metadata from the NBA API.
nba_teams = teams.get_teams()
df_teams = pd.DataFrame(nba_teams)

# Standardize column naming conventions for warehouse integration
df_teams = df_teams.rename(columns={
    'id': 'team_id',
    'full_name': 'team_name',
    'abbreviation': 'team_abbr'
})
df_teams['team_id'] = df_teams['team_id'].astype(str)

# Load foundational team data into BigQuery
client.load_table_from_dataframe(
    df_teams,
    f"{PROJECT_ID}.{DATASET_ID}.dim_teams",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
).result()

print("✅ dim_teams loaded.")

In [ ]:
# --- ENRICHMENT: TEAM HEX COLORS ---
# Purpose: Maps official NBA primary and secondary hex colors to each team.
# This is critical for dynamically coloring the geospatial courts and topographic contours in the visualizations.
color_map = {
    'ATL': ['#E03A3E', '#C1D32F'], 'BOS': ['#007A33', '#BA9653'], 'BKN': ['#000000', '#FFFFFF'],
    'CHA': ['#1D1160', '#00788C'], 'CHI': ['#CE1141', '#000000'], 'CLE': ['#860038', '#041E42'],
    'DAL': ['#00538C', '#002B5E'], 'DEN': ['#0E2240', '#FEC524'], 'DET': ['#C8102E', '#1D428A'],
    'GSW': ['#1D428A', '#FFC72C'], 'HOU': ['#CE1141', '#000000'], 'IND': ['#002D62', '#FDBB30'],
    'LAC': ['#C8102E', '#1D428A'], 'LAL': ['#552583', '#FDB927'], 'MEM': ['#12173F', '#5D76A9'],
    'MIA': ['#98002E', '#F9A01B'], 'MIL': ['#00471B', '#EEE1C6'], 'MIN': ['#0C2340', '#236192'],
    'NOP': ['#002B5C', '#B4975A'], 'NYK': ['#006BB6', '#F58426'], 'OKC': ['#007AC1', '#EF3B24'],
    'ORL': ['#0077C0', '#C4CED4'], 'PHI': ['#006BB6', '#ED174C'], 'PHX': ['#1D1160', '#E56020'],
    'POR': ['#E03A3E', '#000000'], 'SAC': ['#5A2D81', '#63727A'], 'SAS': ['#C4CED4', '#000000'],
    'TOR': ['#CE1141', '#000000'], 'UTA': ['#27104E', '#000000'], 'WAS': ['#002B5C', '#E31837']
}

# Fetch current team table to apply color mapping
df_teams = client.query(f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.dim_teams`").to_dataframe()

# Map the colors using the abbreviation
df_teams['primary_color'] = df_teams['team_abbr'].apply(lambda x: color_map.get(x, ['#808080'])[0])
df_teams['secondary_color'] = df_teams['team_abbr'].apply(lambda x: color_map.get(x, ['#FFFFFF'])[1])

# Reload enriched table
client.load_table_from_dataframe(
    df_teams,
    f"{PROJECT_ID}.{DATASET_ID}.dim_teams",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
).result()

print("✅ dim_teams enriched with primary and secondary hex colors.")

In [ ]:
# --- COMPREHENSIVE DATA WAREHOUSE AUDIT ---
audit_query = f"""
WITH table_stats AS (
    SELECT 'raw_shufinskiy_datanba' as table_name, COUNT(*) as record_count FROM `{PROJECT_ID}.{DATASET_ID}.raw_shufinskiy_datanba`
    UNION ALL
    SELECT 'fct_shots' as table_name, COUNT(*) as record_count FROM `{PROJECT_ID}.{DATASET_ID}.fct_shots`
    UNION ALL
    SELECT 'fct_fouls' as table_name, COUNT(*) as record_count FROM `{PROJECT_ID}.{DATASET_ID}.fct_fouls`
    UNION ALL
    SELECT 'dim_player_seasons' as table_name, COUNT(*) as record_count FROM `{PROJECT_ID}.{DATASET_ID}.dim_player_seasons`
    UNION ALL
    SELECT 'dim_teams' as table_name, COUNT(*) as record_count FROM `{PROJECT_ID}.{DATASET_ID}.dim_teams`
    UNION ALL
    SELECT 'dim_games' as table_name, COUNT(*) as record_count FROM `{PROJECT_ID}.{DATASET_ID}.dim_games`
),
column_stats AS (
    SELECT table_name, COUNT(column_name) as attribute_count
    FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name IN ('raw_shufinskiy_datanba', 'fct_shots', 'fct_fouls', 'dim_player_seasons', 'dim_teams', 'dim_games')
    GROUP BY 1
)
SELECT
    t.table_name,
    t.record_count,
    c.attribute_count
FROM table_stats t
JOIN column_stats c ON t.table_name = c.table_name
ORDER BY t.record_count DESC
"""

inventory_df = client.query(audit_query).to_dataframe()

print("--- FULL DATA WAREHOUSE INVENTORY ---")
display(inventory_df)

# Quick look at the new Game/Team data to ensure no nulls in critical join keys
check_keys_query = f"""
SELECT
    (SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.dim_games` WHERE game_id IS NULL) as null_games,
    (SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.dim_teams` WHERE team_id IS NULL) as null_teams,
    (SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.dim_player_seasons` WHERE player_id IS NULL) as null_players
"""
key_audit = client.query(check_keys_query).to_dataframe()
print("\n--- JOIN KEY INTEGRITY CHECK (SHOULD BE ALL 0) ---")
display(key_audit)